| name | data type | description | notes |
|------|-----------|-------------|-------|
| OBJECTID | OID | Internal feature number.; Description source; ESRI; Description of values Sequential unique whole numbers that are automatically generated. | don't need |
| SHAPE | Geometry | Feature geometry.; Description source; ESRI; Description of values Coordinates defining the features. | point where roads intersect |
| SIFCODE1 | String | Street Code for first intersection retrieved from (SIF); Description source; SIF ||
| SIFCODE2 | String | Street Code for Second intersection retrieved from (SIF); Description source; SIF ||
| INTID | String | ID for the intersection (SCCAD_ID+SIFCODE1+SIFCODE2) | hash for both road IDs |
| SCCAD_ID | Integer | Unique ID number used in Hansen | not needed |
| FST_INTPRE | String | Street Direction (N, S, E, W) for first intersection | human readable info |
| FST_INTNAME | String | Street Name for first intersection | human readable info |
| FST_INTSUF | String | Street Type for first intersection | human readable info |
| SEC_INTPRE | String | Street Direction (N, S, E, W) for second intersection | human readable info |
| SEC_INTNAME | String | Street Name for second intersection | human readable info |
| SEC_INTSUF | String | Street Type for second intersection | human readable info |
| X_COORD | Double | X coordinates for the intersection | more precision? |
| Y_COORD | Double | Y coordinates for the intersection | more precision? |
| FST_SIFID | Integer |  | SIFID: integer value for SIFCODE for 1st street |
| SEC_SIFID | Integer |  | SIFID: integer value for SIFCODE for 2nd street |

In [163]:
# scraping metadata from LOJIC

# imports
import re
import pandas as pd

# constants
metadata_path = "/Users/bencampbell/code/county_coverage/data/raw/intersections/intersections.info"


In [164]:
# parsing centerline metadata 
FIELDNAME = re.compile('\s*Field\s*([A-Z0-9_]+)')
FIELDID = re.compile("Field\s*([A-Z0-9_]+).*")

def parse_fieldname_data(metadata_lines):
    page = list()
    name = 'never appears' 
        
    # break up file to sections 
    for line in metadata_lines:
        line = line.strip()
        m = FIELDID.match(line)
        if m:
            name = m.groups()[0]
            info = list()
            memo = {"name": name, 'info':info}
            page.append(memo)
        elif line.startswith("Hide"):
            assert name in line
        elif line: # ignore empty lines
            info.append(line) 

    # parse infomation in each section
    for section_data in page:
        info = section_data.pop('info')
        index = 0
        for line in info:
            index += 1
            if line.startswith("*\u2009"):
                key, value = line.lstrip("*\u2009").split("\u2003", maxsplit=1)
                section_data[key.lower()] = value
            elif line.startswith("Field description"):
                break
        section_data['description'] = info[index:]
    return page

def get_fieldname_metadata(filepath=metadata_path):
    with open(filepath) as file:
        out = parse_fieldname_data(file)
    return pd.DataFrame.from_dict(out)

intersection_metadata = get_fieldname_metadata()
#all(fieldname_metadata['name'] == fieldname_metadata['alias']) # good news

relevant_columns = ['name', 'data type', 'description']
intersection_meta_filtered = intersection_metadata[columns]

intersection_meta_filtered['description'] = intersection_meta_filtered.description.apply('; '.join)
intersection_meta_filtered

/var/folders/7s/9_9p_dsj0n9581txqt_9rrpc0000gn/T/ipykernel_61505/3440388443.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  intersection_meta_filtered['description'] = intersection_meta_filtered.description.apply('; '.join)


,name,data type,description
0,OBJECTID,OID,Internal feature number.; Description source; ...
1,SHAPE,Geometry,Feature geometry.; Description source; ESRI; D...
2,SIFCODE1,String,Street Code for first intersection retrieved f...
3,SIFCODE2,String,Street Code for Second intersection retrieved ...
4,INTID,String,ID for the intersection (SCCAD_ID+SIFCODE1+SIF...
5,SCCAD_ID,Integer,Unique ID number used in Hansen
6,FST_INTPRE,String,"Street Direction (N, S, E, W) for first inters..."
7,FST_INTNAME,String,Street Name for first intersection
8,FST_INTSUF,String,Street Type for first intersection
9,SEC_INTPRE,String,"Street Direction (N, S, E, W) for second inter..."


In [167]:
# generate markdown

from itertools import chain

def generate_markdown(data):
    page = list()
    data_columns = list(data.columns)
    extra_columns = ['notes']
    add_columns = ' | '.join(extra_columns)
    header = "| {} | {} |".format(' | '.join(data_columns), add_columns)
    blank_line = "| {} |" +  ' | '.join('' for x in extra_columns) + "|"

    
    page.append(header)
    page.append(''.join('-' if char != '|' else char for char in header))
    for _, row in data[data_columns].iterrows():
        g = blank_line.format(' | '.join(row)) # all row values are strings, 
        #                                      # so I did not need to cast values to str here
        page.append(g)
        #print(g)
    return page

page = generate_markdown(intersection_meta_filtered)

print(*page, sep='\n')

| name | data type | description | notes |
|------|-----------|-------------|-------|
| OBJECTID | OID | Internal feature number.; Description source; ESRI; Description of values Sequential unique whole numbers that are automatically generated. ||
| SHAPE | Geometry | Feature geometry.; Description source; ESRI; Description of values Coordinates defining the features. ||
| SIFCODE1 | String | Street Code for first intersection retrieved from (SIF); Description source; SIF ||
| SIFCODE2 | String | Street Code for Second intersection retrieved from (SIF); Description source; SIF ||
| INTID | String | ID for the intersection (SCCAD_ID+SIFCODE1+SIFCODE2) ||
| SCCAD_ID | Integer | Unique ID number used in Hansen ||
| FST_INTPRE | String | Street Direction (N, S, E, W) for first intersection ||
| FST_INTNAME | String | Street Name for first intersection ||
| FST_INTSUF | String | Street Type for first intersection ||
| SEC_INTPRE | String | Street Direction (N, S, E, W) for second intersecti

In [ ]:
# metadata info about X, Y coordinates

# via https://www.lojic.org/sites/default/files/metadata/address_intersection.htm

#Extent 
#Geographic extent 
#Bounding rectangle 
#Extent type  Extent used for searching
west_longitude = -85.945347
east_longitude = -85.344499
north_latitude = 38.378034
south_latitude = 38.005894
#* Extent contains the resource Yes

#Extent in the item's coordinate system 
west_longitude_co = 1154395.500000
east_longitude_co = 1325086.990000
south_latitude_co = 188677.437500
north_latitude_co = 321629.781250
#* Extent contains the resource Yes